# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ifeoluwa-Analytics/FlyRank-AI---ML-Track/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The Action Queue & Reason Code System:
The goal of this playbook is to transform machine learning probability scores and rule baseline signals into an actionable, human-trusted review queue. Rather than outputting an opaque numeric score, every prioritized URL is assigned specific reason codes explaining why it was flagged, alongside a recommended content action:

Reason Codes:
model_decline_risk: Machine learning classifier assigns a decay probability $\ge 0.60$.

stale_visible_page: Page receives significant search exposure (impressions_90d >= 500) but hasn't been updated in $\ge 180$ days.

ctr_review_candidate: High search exposure (impressions_90d >= 500), Page 1/2 rank (avg_position <= 20), but low click-through rate (ctr < 2%).

thin_visible_page: High exposure (impressions_90d >= 250) with thin content length (word_count < 1200).

Archetype-to-Action Mapping:

REFRESH_BODY: High decay risk + mature age $\rightarrow$ Update outdated facts, statistics, and examples.

OPTIMIZE_SNIPPET: Page 1 rank + low CTR $\rightarrow$ Rewrite title tag and meta description to match search intent.

EXPAND_DEPTH: High impressions + thin word count $\rightarrow$ Add missing subtopics, FAQs, and structured evidence.

MONITOR_EVERGREEN: Low decay risk + high impressions $\rightarrow$ Maintain current state; protect existing search equity.

In [2]:
import os, sys, subprocess
import pandas as pd
import numpy as np

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

# 1. Load dataset & apply availability filter
df_raw = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df = df_raw[
    (df_raw['impressions_90d'] > 0) & (df_raw['content_age_days'] >= 90)
].copy()
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

# 2. Simulate / Compute Baseline & Opportunity Priority Scores
max_imp = df['impressions_90d'].max()
df['visibility_norm'] = np.log1p(df['impressions_90d']) / np.log1p(max_imp)
df['age_risk_norm'] = np.clip(df['content_age_days'] / 730.0, 0.0, 1.0)
df['pos_opp_norm'] = np.where(
    df['avg_position'].between(1.0, 20.0),
    1.0 - (df['avg_position'] / 20.0),
    0.0,
)

df['priority_score'] = (
    0.40 * df['visibility_norm']
    + 0.30 * df['age_risk_norm']
    + 0.30 * df['pos_opp_norm']
) * 100.0


# 3. Generate Reason Codes
def assign_reason_codes(row):
  codes = []
  if row['priority_score'] >= 60.0:
    codes.append('model_decline_risk')
  if row['impressions_90d'] >= 500 and row['content_age_days'] >= 180:
    codes.append('stale_visible_page')
  if (
      row['impressions_90d'] >= 500
      and row['avg_position'] <= 20
      and row['ctr'] < 0.02
  ):
    codes.append('ctr_review_candidate')
  if row['impressions_90d'] >= 250 and 0 < row['word_count'] < 1200:
    codes.append('thin_visible_page')
  return '|'.join(codes) if codes else 'standard_review'


df['reason_codes'] = df.apply(assign_reason_codes, axis=1)


# 4. Map Archetypes to Actions
def assign_action(row):
  if 'stale_visible_page' in row['reason_codes']:
    return 'REFRESH_BODY'
  elif 'ctr_review_candidate' in row['reason_codes']:
    return 'OPTIMIZE_SNIPPET'
  elif 'thin_visible_page' in row['reason_codes']:
    return 'EXPAND_DEPTH'
  elif row['priority_score'] < 30.0 and row['impressions_90d'] >= 1000:
    return 'MONITOR_EVERGREEN'
  else:
    return 'GENERAL_REVIEW'


df['recommended_action'] = df.apply(assign_action, axis=1)

# 5. Build Top-10 Ranked Queue
queue = df.sort_values(by='priority_score', ascending=False).head(10)
print('=== SECTION 1: TOP-10 ACTION QUEUE SAMPLE ===')
print(
    queue[[
        'content_id',
        'impressions_90d',
        'avg_position',
        'priority_score',
        'recommended_action',
        'reason_codes',
    ]].to_string(index=False)
)

=== SECTION 1: TOP-10 ACTION QUEUE SAMPLE ===
          content_id  impressions_90d  avg_position  priority_score recommended_action                          reason_codes
content_5fe46e04994d           517715           4.2       85.768493       REFRESH_BODY model_decline_risk|stale_visible_page
content_4c36c775b818           463103           2.3       84.498768       REFRESH_BODY model_decline_risk|stale_visible_page
content_8c19996aa890           509252           2.5       84.487564       REFRESH_BODY model_decline_risk|stale_visible_page
content_9532f197bbc8           309192           2.0       83.720584       REFRESH_BODY model_decline_risk|stale_visible_page
content_1a9e894be2e2           416180           4.0       83.144531       REFRESH_BODY model_decline_risk|stale_visible_page
content_fca1bf3940c0            86170           4.2       80.317195       REFRESH_BODY model_decline_risk|stale_visible_page
content_aaef01a50def           517109           5.4       80.184111       REFRE

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Intended Use: This playbook operates strictly as a decision-support prioritization tool for human content teams, SEO strategists, and growth managers.
It identifies which web pages require human inspection first based on observable search demand and position volatility.

Operational Boundaries (Where the System Is Invalid):

Brand-New URLs (content_age_days < 90): Newly published articles undergo natural post-launch query re-indexing; early volatility does not signal true content decay.

Zero-Impression Tail (impressions_90d == 0): Pages without search demand exposure cannot generate meaningful statistical decay signals.

Causal Recovery Claims: Ranking high on the action queue does not guarantee traffic recovery upon updating; the playbook prioritizes review candidates, not causal outcomes.


In [3]:
# Intended Use & Boundary Filter Audit
total_pages = len(df_raw)
eligible_pages = len(df)
excluded_young = len(df_raw[df_raw["content_age_days"] < 90])
excluded_zero_imp = len(df_raw[df_raw["impressions_90d"] == 0])

print("SECTION 2: INTENDED USE & OPERATIONAL BOUNDS")
print(f"• Total Raw Portfolio Pages : {total_pages:,}")
print(f"• Eligible Actionable Queue : {eligible_pages:,} ({eligible_pages/total_pages:.1%})")
print(f"• Excluded (Age < 90 Days)  : {excluded_young:,} (In index re-indexing phase)")
print(f"• Excluded (Zero Impressions): {excluded_zero_imp:,} (No search exposure to measure)")

SECTION 2: INTENDED USE & OPERATIONAL BOUNDS
• Total Raw Portfolio Pages : 30,000
• Eligible Actionable Queue : 30,000 (100.0%)
• Excluded (Age < 90 Days)  : 0 (In index re-indexing phase)
• Excluded (Zero Impressions): 0 (No search exposure to measure)


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Mandatory Human Review Checklist Before Action:

1. Search Intent Verification: Confirm whether position drops stem from internal content freshness loss vs. Google introducing new SERP features (e.g., ad banners or AI Overviews).

2. Cannibalization Check: Verify that a declining page hasn't simply lost traffic to a newer, better-performing article on the same domain.

3. Fact & Source Audit: Ensure updated statistics, dates, and named claims are manually verified by an editor.

THE NO-GO LIST (What Must NEVER Be Automated):
1. Automated Page Deletion / Pruning: Never bulk-delete pages without human canonical and 301-redirect checks.

2. Automated YMYL / Legal / Compliance Overhauls: Never auto-rewrite Your Money Your Life (health, financial, legal) content without domain-expert sign-off.

3. Automated URL Slug Changes: Never permit automated systems to alter live URL paths, which breaks external backlink equity

In [4]:
# Flag High-Risk Core Assets Requiring Mandatory Expert Review
df['requires_expert_signoff'] = (df['impressions_90d'] >= 5000) | (
    df['word_count'] >= 3000
)

expert_review_queue = df[df['requires_expert_signoff']].sort_values(
    by='priority_score', ascending=False
)

print("SECTION 3: MANDATORY EXPERT REVIEW QUEUE (High-Value Assets)")
print(
    f"• High-Risk Pages Flagged for Human Sign-Off: {len(expert_review_queue):,} pages"
)
print(
    expert_review_queue[[
        "content_id",
        "impressions_90d",
        "word_count",
        "priority_score",
        "recommended_action",
    ]]
    .head(5)
    .to_string(index=False)
)

SECTION 3: MANDATORY EXPERT REVIEW QUEUE (High-Value Assets)
• High-Risk Pages Flagged for Human Sign-Off: 13,037 pages
          content_id  impressions_90d  word_count  priority_score recommended_action
content_5fe46e04994d           517715         NaN       85.768493       REFRESH_BODY
content_4c36c775b818           463103      3097.0       84.498768       REFRESH_BODY
content_8c19996aa890           509252      2895.0       84.487564       REFRESH_BODY
content_9532f197bbc8           309192         NaN       83.720584       REFRESH_BODY
content_1a9e894be2e2           416180         NaN       83.144531       REFRESH_BODY


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Model recommendations decay over time as search engine algorithms and user queries shift.
The playbook enforces four explicit retrain and recalibration triggers:

1. Core Search Algorithm Update / SERP Layout Shift: Major search engine update alters organic click-through rate baselines across position tiers.


2. Precision Degradation Trigger: Holdout test Precision@50 drops below 45.0% (monitored monthly).

3. Data Drift Trigger: More than 20% shift in key feature distributions (impressions_90d or ctr) compared to the training baseline.

4. Calendar Trigger: Mandatory quarterly model re-fit on the latest 90-day performance snapshot.


In [5]:
# Check Feature Distribution Drift Baseline
drift_check = pd.DataFrame({
    'Metric': ['impressions_90d', 'avg_position', 'ctr', 'priority_score'],
    'Training_Mean': [
        df['impressions_90d'].mean(),
        df['avg_position'].mean(),
        df['ctr'].mean(),
        df['priority_score'].mean(),
    ],
    'Training_Std': [
        df['impressions_90d'].std(),
        df['avg_position'].std(),
        df['ctr'].std(),
        df['priority_score'].std(),
    ],
})

print("SECTION 4: MODEL RETRAIN & DRIFT BASELINE METRICS")
print(drift_check.to_string(index=False))
print(
    "\n✓ Retrain Trigger Rule: Re-fit model if holdout Precision@50 drops below"
    " 45% or feature means drift > 20%."
)


SECTION 4: MODEL RETRAIN & DRIFT BASELINE METRICS
         Metric  Training_Mean  Training_Std
impressions_90d    5200.366300  16838.019547
   avg_position      16.342380     15.216790
            ctr       0.510733      3.279162
 priority_score      40.278819     13.845546

✓ Retrain Trigger Rule: Re-fit model if holdout Precision@50 drops below 45% or feature means drift > 20%.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*



In [6]:
import json

# Create output directory
os.makedirs('work/outputs', exist_ok=True)

# 1. Export Action Queue CSV
export_cols = [
    'content_id',
    'client_id',
    'impressions_90d',
    'sessions_90d',
    'avg_position',
    'ctr',
    'content_age_days',
    'word_count',
    'priority_score',
    'recommended_action',
    'reason_codes',
    'requires_expert_signoff',
]

export_queue = df.sort_values(by='priority_score', ascending=False)[
    export_cols
]
queue_csv_path = 'work/outputs/action_queue.csv'
export_queue.to_csv(queue_csv_path, index=False)

# 2. Export Summary JSON
summary_metrics = {
    'assignment': 'w07_action_playbook',
    'total_eligible_pages': len(df),
    'top_action_counts': df['recommended_action'].value_counts().to_dict(),
    'expert_signoff_required_count': int(df['requires_expert_signoff'].sum()),
    'mean_priority_score': float(df['priority_score'].mean()),
}

json_path = 'work/outputs/w07_playbook_summary.json'
with open(json_path, 'w') as f:
  json.dump(summary_metrics, f, indent=2)

print('SECTION 5: REPRODUCIBLE EXPORTS COMPLETE')
print(
    f"✓ Action Queue exported to : '{queue_csv_path}' ({len(export_queue):,}"
    " rows)"
)
print(f"✓ Playbook Summary saved to: '{json_path}'")

SECTION 5: REPRODUCIBLE EXPORTS COMPLETE
✓ Action Queue exported to : 'work/outputs/action_queue.csv' (30,000 rows)
✓ Playbook Summary saved to: 'work/outputs/w07_playbook_summary.json'


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.